# TP · Identidad y emoción de actores argentinos — Grupo 6
**IAyAA II · Licenciatura en Ciencia de Datos · UCA Rosario · 2026**

Martín Masuelli · Francisco Miles · Tomás Lizazu · Bautista Verna · Juan Diego Ancarani · Manuel Lafosse

Dos tareas sobre el dataset del curso (recortes de rostro 224×224):
- **Identidad:** ¿quién es? (20 actores en la versión actual del dataset)
- **Emoción:** ¿qué gesto hace? (alegría, enojo, neutral, sorpresa, tristeza)

| # | Sección | Pregunta que responde |
|---|---|---|
| 1 | Entorno | ¿Cómo se reproduce exactamente este notebook? |
| 2 | Datos | ¿Qué hay en el dataset y qué le falta? |
| 3 | Partición | ¿Cómo evitamos que el modelo "reconozca el video" en vez del gesto? |
| 4 | Baselines | ¿Contra qué piso medimos? |
| 5 | Acta de límites | ¿Por qué hace falta una red convolucional? |
| 6 | Pipeline y aumentación | ¿Cómo se alimenta la red y qué invarianzas le enseñamos? |
| 7 | CNN propia | ¿Cuánto rinde una red entrenada desde cero con pocos datos? |
| 8 | ResNet50 (transferencia) | ¿Cuánto aportan ImageNet y el ajuste fino? |
| 9 | Evaluación final en test | Resultado honesto, evaluado **una sola vez** |
| 10 | Interpretación | ¿El modelo mira la cara o el fondo? |
| 11 | Conclusiones y limitaciones | |

**Reglas que respeta el notebook:** el Drive del curso se monta en **solo lectura**; la partición es **por video**; toda decisión se toma contra **validación** y el test se evalúa una única vez al final; semillas fijas.

El código está versionado en `github.com/FMiles-2006/tp-emociones-g6` (historial de cambios y bitácora de decisiones `docs/BITACORA.md`). **Requisitos:** entorno con **GPU T4** y, en "Mi unidad", un acceso directo a la carpeta `CARPETA COMPARTIDA PARA TODOS`.

## 1. Entorno reproducible
Todo valor que afecta un resultado (semilla, clases, tamaños, fracciones de la partición) se define **una sola vez** en la celda siguiente. La semilla se fija en Python, NumPy y TensorFlow (y se piden operaciones deterministas). En GPU puede quedar una variación mínima entre corridas por algunas operaciones de cuDNN: se declara en vez de esconderla.

In [ ]:
# 1.1 Librerías, configuración y semillas
import os, re, glob, json, time, random, shutil, hashlib, unicodedata
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display

import tensorflow as tf
import keras
from keras import layers
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

# --- Configuración (única fuente de verdad) ----------------------------------
SEED = 42
EMOCIONES = ("alegria", "enojo", "neutral", "sorpresa", "tristeza")  # vocabulario cerrado del curso (sin Miedo)
IMG_SIZE = 224             # tamaño de los recortes = entrada nativa de ResNet50
IMG_SIZE_BASELINE = 64     # resolución reducida para los baselines clásicos
FRACCIONES = {"train": 0.70, "val": 0.15, "test": 0.15}
BATCH_SIZE = 32
CSV_MASTER, CARPETA_IMAGENES = "etiquetas_master_final.csv", "Imagenes_Finales"

# Identidades que el CSV del curso escribe distinto
MAPA_ACTORES = {
    "tini": "tini_stoessel", "georginabarbarossa": "georgina_barbarossa",
    "oscarmartinez": "oscar_martinez", "juliochavez": "julio_chavez",
    "pachupena": "pachu_pena", "rodrigodelaserna": "rodrigo_de_la_serna",
    "francella": "guillermo_francella", "darin": "ricardo_darin", "lali": "lali_esposito",
    "milei": "javier_milei", "barassi": "dario_barassi", "momo_benavides": "momo",
    "geronimo_benavides": "momo", "flavio_azzaro": "azzaro",
    "franco_colapinto": "colapinto", "lionel_scaloni": "scaloni",
}


def fijar_semillas(seed=SEED):
    # Fija todas las fuentes de azar. Se llama al inicio y antes de construir cada modelo.
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


fijar_semillas()
sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 3)
print(f"TensorFlow {tf.__version__} · Keras {keras.__version__} · scikit-learn {sklearn.__version__} · NumPy {np.__version__}")
print("GPU:", tf.config.list_physical_devices("GPU") or "NO — activar GPU T4 en Entorno de ejecución")

In [ ]:
# 1.2 Datos: Drive del curso en SOLO LECTURA + copia local de trabajo
from google.colab import drive
drive.mount("/content/drive", readonly=True)   # readonly: es imposible modificar el Drive del curso

candidatas = (glob.glob("/content/drive/MyDrive/CARPETA COMPARTIDA PARA TODOS")
              + glob.glob("/content/drive/MyDrive/*/CARPETA COMPARTIDA PARA TODOS"))
assert candidatas, "Falta el acceso directo a 'CARPETA COMPARTIDA PARA TODOS' en Mi unidad."
CARPETA_CURSO = Path(candidatas[0])
DATASET = Path("/content/datos/DATASET_FINAL")
if not DATASET.exists():   # la copia local es más rápida y aísla el original
    shutil.copytree(CARPETA_CURSO / "DATASET_FINAL", DATASET, ignore=shutil.ignore_patterns("Metricas_Dataset"))
print("Carpeta del curso (solo lectura):", CARPETA_CURSO)
print("Copia de trabajo:", DATASET, "→", len(os.listdir(DATASET / CARPETA_IMAGENES)), "imágenes")

## 2. Los datos

Al auditar el dataset del curso encontramos tres problemas que el código resuelve:
1. **Nombres inconsistentes**: tildes, mayúsculas y la `ñ` codificada como `#U00f1` por las descargas de Drive. Se cruzan por una *clave* normalizada.
2. **Identidades escritas distinto** (`tini` / `tini_stoessel`): se unifican con `MAPA_ACTORES`.
3. **El CSV maestro no tiene el video de origen**, que es indispensable para la partición. Se recupera de los registros que subió cada grupo a su carpeta (G1, G3, G4, G6, G7, G8) y, si una imagen no figura, del patrón de su nombre (`actor__video__f123.jpg`, `Cande1_frame_10.jpg`, `momo_video_2_0012.jpg`). Último recurso conservador: el nombre sin números (agrupa de más; nunca genera fugas).

In [ ]:
# 2.1 Normalización de nombres y recuperación del video de origen
def clave_archivo(nombre):
    # 'PachuPe#U00f1a_Alegr#U00eda.jpg' y 'PachuPeña_Alegría.jpg' -> 'pachupena_alegria.jpg'
    s = re.sub(r"#U([0-9a-fA-F]{4})", lambda m: chr(int(m.group(1), 16)), str(nombre))
    s = unicodedata.normalize("NFKD", s)
    return "".join(c for c in s if not unicodedata.combining(c)).lower().strip()


def normalizar_etiqueta(texto):
    s = re.sub(r"[\s\-_]+", "_", clave_archivo(texto))
    return re.sub(r"[^a-z0-9_]", "", s).strip("_")


def normalizar_actor(nombre):
    actor = normalizar_etiqueta(nombre)
    return MAPA_ACTORES.get(actor, actor)


PATRONES_VIDEO = (
    re.compile(r"^.+?__(?P<video>.+?)__f\d+\.jpe?g$"),          # G1/G10: actor__video__f0000123.jpg
    re.compile(r"^(?P<video>.+?)_frame_\d+\.jpe?g$"),           # G4: Cande1_frame_000010.jpg
    re.compile(r"^(?P<video>.+?)(?:_p\d+)?_\d+\.jpe?g$"),       # G7: momo_video_1_p2_0003.jpg (p2 = parte del mismo clip)
)


def video_desde_nombre(clave):
    for patron in PATRONES_VIDEO:
        m = patron.match(clave)
        if m:
            return m.group("video")
    return None


def leer_registros_grupos(carpeta_curso):
    # Mapa clave -> video a partir del registro por imagen que subió cada grupo (solo lectura).
    fuentes = []
    cols = ["archivo", "actor", "emocion", "video", "escena"]
    for g in ("G3", "G7", "G8"):
        p = carpeta_curso / g / "etiquetas_master.txt"
        if p.exists():
            d = pd.read_csv(p, sep=";", header=None, names=cols)
            fuentes.append(d[["archivo", "video"]])
    especiales = {"G1": ("registro_grupo.csv", "archivo", "video", {}),
                  "G4": ("labels_224_final_grupo4.csv", "imagen", "video", {}),
                  "G6": ("DATASET_FINAL.csv", "imagen", "video_origen", {"encoding": "utf-8-sig"})}
    for g, (nombre, c_arch, c_vid, kw) in especiales.items():
        p = carpeta_curso / g / nombre
        if p.exists():
            d = pd.read_csv(p, **kw).dropna(subset=[c_arch, c_vid])
            fuentes.append(pd.DataFrame({"archivo": d[c_arch], "video": d[c_vid].astype(str).str.slice(0, 24)}))
    if not fuentes:
        return {}
    todo = pd.concat(fuentes, ignore_index=True)
    todo["video"] = todo["video"].astype(str).str.replace(r"\.mp4$", "", regex=True)
    todo["clave"] = todo["archivo"].map(clave_archivo)
    return dict(todo.drop_duplicates("clave").set_index("clave")["video"])


def asignar_video(claves, registro):
    videos, origenes = [], []
    for c in claves:
        if c in registro:
            videos.append(registro[c]); origenes.append("registro del grupo")
        elif (v := video_desde_nombre(c)) is not None:
            videos.append(v); origenes.append("nombre de archivo")
        else:
            videos.append(re.sub(r"\d+", "", Path(c).stem).strip("_")); origenes.append("prefijo (conservador)")
    return videos, origenes

In [ ]:
# 2.2 Registro: una fila por imagen utilizable (nunca escribe en el Drive)
def huella_dataset(df):
    # Hash de qué imágenes hay y con qué etiquetas: identifica la VERSIÓN del dataset.
    contenido = "\n".join(sorted(df["clave"] + ";" + df["actor"] + ";" + df["emocion"]))
    return hashlib.sha256(contenido.encode()).hexdigest()[:12]


def construir_registro(ruta_dataset, carpeta_curso):
    df = pd.read_csv(ruta_dataset / CSV_MASTER, sep=None, engine="python", encoding="utf-8-sig")
    df.columns = [c.strip().lower() for c in df.columns]
    fisicas = {clave_archivo(p.name): p for p in (ruta_dataset / CARPETA_IMAGENES).iterdir()}
    df = df.assign(clave=df["archivo"].map(clave_archivo), actor=df["actor"].map(normalizar_actor),
                   emocion=df["emocion"].map(normalizar_etiqueta))
    rep = {"filas en el CSV": len(df), "imágenes en la carpeta": len(fisicas)}
    n = len(df); df = df[df["emocion"].isin(EMOCIONES)]; rep["descartadas: emoción inválida"] = n - len(df)
    n = len(df); df = df.drop_duplicates("clave"); rep["descartadas: duplicadas"] = n - len(df)
    n = len(df); df = df[df["clave"].isin(fisicas)]; rep["descartadas: sin imagen física"] = n - len(df)
    df = df.assign(ruta=df["clave"].map(lambda c: str(fisicas[c])))
    videos, origenes = asignar_video(df["clave"], leer_registros_grupos(carpeta_curso))
    df["video"] = df["actor"] + "|" + pd.Series(videos, index=df.index).astype(str)   # el video se califica con el actor
    df["origen_video"] = origenes
    df = df.sort_values("clave").reset_index(drop=True)[["archivo", "clave", "ruta", "actor", "emocion", "video", "origen_video"]]
    rep["imágenes utilizables"] = len(df)
    return df, rep


df, reporte_carga = construir_registro(DATASET, CARPETA_CURSO)
HUELLA = huella_dataset(df)
for k, v in reporte_carga.items():
    print(f"  {k:32s} {v:>6}")
print(f"  {'huella (versión) del dataset':32s} {HUELLA}")
print("  origen del video:", df["origen_video"].value_counts().to_dict())
print(f"  actores: {df['actor'].nunique()} · videos: {df['video'].nunique()}")

In [ ]:
# 2.3 Distribución actor × emoción (tabla del enunciado) y gráficos
display(pd.crosstab(df["actor"], df["emocion"], margins=True, margins_name="Total"))

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 2]})
df["emocion"].value_counts().reindex(EMOCIONES).plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Imágenes por emoción"); axes[0].set_ylabel("imágenes")
df["actor"].value_counts().plot.bar(ax=axes[1], color="slategray")
axes[1].set_title("Imágenes por actor")
plt.tight_layout(); plt.show()

In [ ]:
# 2.4 Ejemplos: una fila por emoción
fig, axes = plt.subplots(len(EMOCIONES), 8, figsize=(14, 9))
for f, emo in enumerate(EMOCIONES):
    for c, (_, fila) in enumerate(df[df["emocion"] == emo].sample(8, random_state=SEED).iterrows()):
        axes[f, c].imshow(Image.open(fila["ruta"])); axes[f, c].axis("off")
        axes[f, c].set_title(fila["actor"].split("_")[0], fontsize=7)
    axes[f, 0].set_title(f"{emo.upper()}\n{axes[f, 0].get_title()}", fontsize=8, loc="left")
plt.suptitle("Ejemplos por emoción"); plt.tight_layout(); plt.show()

**Lectura del dataset (insumo de la ficha):**
- El desbalance de emociones es moderado: *neutral* es la más frecuente y *tristeza* la más escasa. Por eso reportamos **F1 macro** además de accuracy y entrenamos con **pesos por clase**.
- Identidad: la mayoría de los actores tiene ~100 imágenes; Francella y Tini tienen más porque aparecen en dos grupos.
- **Limitaciones:** Pachu Peña (122 imágenes del G6) no entró al dataset del curso por la "ñ" del nombre de archivo; Rodrigo de la Serna no tiene imágenes subidas; los grupos 2, 5 y 9 no aportaron. Los recortes del G6 son centrales (la cara ocupa menos del cuadro).
- **Techo realista para emoción:** el acuerdo entre etiquetadores de nuestro control cruzado del TP1 fue **42,6 %**. Un modelo no puede ser creíblemente "más preciso" que el acuerdo humano sobre las propias etiquetas.

## 3. Partición por video (la regla de oro)

Dos cuadros del mismo plano son casi idénticos (en la auditoría encontramos ~1.500 pares casi duplicados dentro de un mismo video). Si se reparten entre train y test, el modelo **reconoce el plano y no el gesto**, y el resultado se infla.

- **R1 (sin excepciones):** ningún video de train aparece en validación ni en test. Se verifica con un `assert`.
- **R2:** se parte por actor, para que los 20 actores estén en train (indispensable para identidad) y también en val y test.
- **R3:** por actor se eligen los videos de train buscando ~70 % de sus imágenes (búsqueda exhaustiva con ≤ 16 videos), prefiriendo dejar para evaluación videos que se puedan separar entre val y test; los videos restantes se reparten entre val y test **por video**.
- **R4 (concesión documentada):** cuando no se puede (actores con 2 o 3 videos muy desparejos), val y test comparten **un** video cortado en dos mitades contiguas. Train nunca comparte.
- **R5:** un actor con un solo video va entero a train y se reporta.
- **Determinista:** los empates se rompen con un hash del nombre del video + la semilla.

In [ ]:
# 3.1 Funciones de partición
SPLITS = ("train", "val", "test")
MAX_EXHAUSTIVO, MIN_FRAC_EVAL, PENALIDAD_R4 = 16, 0.35, 0.10


def _h(nombre):
    return hashlib.sha256(f"{SEED}:{nombre}".encode()).hexdigest()


def _separable_por_video(tamanios):
    n, total = len(tamanios), tamanios.sum()
    if n < 2:
        return False
    if n > MAX_EXHAUSTIVO:
        return True
    v = tamanios.to_numpy()
    return any(min(s, total - s) >= MIN_FRAC_EVAL * total
               for s in (sum(v[i] for i in range(n) if m >> i & 1) for m in range(1, 2 ** n - 1)))


def _elegir_train(tamanios):
    videos = sorted(tamanios.index, key=lambda v: (-tamanios[v], _h(v)))
    total = tamanios.sum(); objetivo = FRACCIONES["train"] * total
    if len(videos) <= MAX_EXHAUSTIVO:
        mejor, mejor_costo = None, np.inf
        for m in range(1, 2 ** len(videos) - 1):
            elegidos = [v for i, v in enumerate(videos) if m >> i & 1]
            costo = abs(tamanios[elegidos].sum() - objetivo) / total
            if not _separable_por_video(tamanios.drop(elegidos)):
                costo += PENALIDAD_R4
            if costo < mejor_costo:
                mejor, mejor_costo = set(elegidos), costo
        return mejor
    elegidos, acum = set(), 0
    for v in videos:
        if acum + tamanios[v] <= objetivo or not elegidos:
            elegidos.add(v); acum += tamanios[v]
    return elegidos - ({videos[-1]} if len(elegidos) == len(videos) else set())


def _repartir_evaluacion(g):
    tamanios = g.groupby("video").size()
    videos = sorted(tamanios.index, key=_h); total = tamanios.sum()
    mejor, mejor_costo = None, np.inf
    if 2 <= len(videos) <= MAX_EXHAUSTIVO:
        for m in range(1, 2 ** len(videos) - 1):
            a_val = [v for i, v in enumerate(videos) if m >> i & 1]
            n_val = tamanios[a_val].sum()
            if min(n_val, total - n_val) >= MIN_FRAC_EVAL * total and abs(n_val - total / 2) < mejor_costo:
                mejor, mejor_costo = set(a_val), abs(n_val - total / 2)
    if mejor is not None:                                     # R3: por video
        return pd.Series(np.where(g["video"].isin(mejor), "val", "test"), index=g.index)
    orden = g.assign(_hv=g["video"].map(_h)).sort_values(["_hv", "clave"]).index   # R4: mitades contiguas
    mitad = len(orden) // 2
    return pd.Series(["val"] * mitad + ["test"] * (len(orden) - mitad), index=orden).reindex(g.index)


def particionar(df):
    split, avisos = pd.Series(index=df.index, dtype=object), []
    for actor, g in df.groupby("actor", sort=True):
        tamanios = g.groupby("video").size()
        if len(tamanios) == 1:                                # R5
            split.loc[g.index] = "train"; avisos.append(f"{actor}: un solo video -> todo a train (R5)"); continue
        es_train = g["video"].isin(_elegir_train(tamanios))
        split.loc[g.index[es_train]] = "train"
        ev = _repartir_evaluacion(g[~es_train]); split.loc[ev.index] = ev
        compartidos = set(g.loc[ev.index[ev == "val"], "video"]) & set(g.loc[ev.index[ev == "test"], "video"])
        if compartidos:
            avisos.append(f"{actor}: val y test comparten {sorted(compartidos)} (R4)")
    return split, avisos


def verificar_sin_fugas(df, split):
    vids = {s: set(df.loc[split == s, "video"]) for s in SPLITS}
    fuga = vids["train"] & (vids["val"] | vids["test"])
    assert not fuga, f"FUGA: {sorted(fuga)[:5]}"
    assert split.notna().all(), "Hay imágenes sin split"
    assert set(df["actor"]) == set(df.loc[split == "train", "actor"]), "Hay actores sin train"


def resumen_particion(df, split):
    filas = []
    for s in SPLITS:
        x = df[split == s]; dist = x["emocion"].value_counts(normalize=True)
        filas.append({"split": s, "imágenes": len(x), "%": round(100 * len(x) / len(df), 1),
                      "videos": x["video"].nunique(), "actores": x["actor"].nunique(),
                      **{e: f"{100 * dist.get(e, 0):.1f}%" for e in EMOCIONES}})
    return pd.DataFrame(filas).set_index("split")

In [ ]:
# 3.2 Partición del dataset
split, avisos = particionar(df)
verificar_sin_fugas(df, split)
assert split.equals(particionar(df)[0]), "La partición no es determinista"
df["split"] = split
print("✔ Sin fugas: ningún video de train aparece en validación o test. ✔ Determinista.\n")
display(resumen_particion(df, split))
print("\nActores con la concesión R4 (val y test comparten un video; train no):")
for a in avisos:
    print("  -", a)
display(pd.crosstab(df["actor"], df["split"])[list(SPLITS)])

## 4. Baselines: el piso contra el que se mide todo

Ninguna red se justifica sin un baseline (NB01). Tres baselines, medidos **en validación** (test queda reservado):
- **Trivial:** predice siempre la clase más frecuente de train.
- **Random Forest** y **MLP de scikit-learn** sobre píxeles: imagen en gris reducida a 64×64 → 4.096 columnas. Se reduce porque con 224×224×3 = 150.528 columnas y ~1.900 imágenes no hay de dónde generalizar.

In [ ]:
# 4.1 Funciones de baselines
def cargar_pixeles(rutas, tam=IMG_SIZE_BASELINE):
    X = np.empty((len(rutas), tam * tam), dtype=np.float32)
    for i, r in enumerate(rutas):
        with Image.open(r) as im:
            X[i] = np.asarray(im.convert("L").resize((tam, tam), Image.BILINEAR), dtype=np.float32).ravel() / 255.0
    return X


def metricas(y_true, y_pred):
    # F1 macro pesa igual a cada clase: con clases desbalanceadas es la métrica honesta.
    return {"accuracy": accuracy_score(y_true, y_pred), "f1_macro": f1_score(y_true, y_pred, average="macro")}


def a_enteros(y_tr, y_ev):
    # scikit-learn 1.6 (Colab) falla en MLPClassifier(early_stopping=True) con etiquetas de texto.
    clases = {c: i for i, c in enumerate(np.unique(np.concatenate([np.asarray(y_tr), np.asarray(y_ev)])))}
    return np.array([clases[v] for v in y_tr]), np.array([clases[v] for v in y_ev])


def evaluar_baselines(X_tr, y_tr, X_ev, y_ev):
    y_tr, y_ev = a_enteros(y_tr, y_ev)
    modelos_cl = {
        "Trivial (clase más frecuente)": DummyClassifier(strategy="most_frequent"),
        "Random Forest (píxeles 64x64)": RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=SEED),
        "MLP scikit-learn (píxeles 64x64)": MLPClassifier(hidden_layer_sizes=(256,), early_stopping=True, max_iter=200, random_state=SEED),
    }
    filas = []
    for nombre, m in modelos_cl.items():
        t0 = time.perf_counter(); m.fit(X_tr, y_tr)
        filas.append({"modelo": nombre, **metricas(y_ev, m.predict(X_ev)), "segundos": round(time.perf_counter() - t0, 1)})
    return pd.DataFrame(filas).set_index("modelo")

In [ ]:
# 4.2 Baselines para las dos tareas (VALIDACIÓN)
X_pix = cargar_pixeles(df["ruta"])
m_tr, m_va = (df["split"] == "train").to_numpy(), (df["split"] == "val").to_numpy()
resultados_baseline = {}
for tarea in ("actor", "emocion"):
    resultados_baseline[tarea] = evaluar_baselines(X_pix[m_tr], df.loc[m_tr, tarea], X_pix[m_va], df.loc[m_va, tarea])
    print(f"\n=== {tarea.upper()} (validación)"); display(resultados_baseline[tarea])

## 5. Acta de límites (NB02) con nuestro dataset

1. **Costo:** al agrandar el MLP, el tiempo de entrenamiento crece mucho más rápido que el beneficio.
2. **Permutación de píxeles:** se mezclan los píxeles de *todas* las imágenes con una misma permutación fija. Para un humano quedan irreconocibles. Si el MLP rinde igual, nunca usó la estructura espacial de la cara: la trata como una tabla de números sueltos.

In [ ]:
# 5.1 Mediciones del acta
def permutacion_fija(n):
    return np.random.default_rng(SEED).permutation(n)


def medir_costo_mlp(X_tr, y_tr, X_ev, y_ev, arqs=((64,), (256,), (512, 256), (1024, 512))):
    y_tr, y_ev = a_enteros(y_tr, y_ev); filas = []
    for arq in arqs:
        m = MLPClassifier(hidden_layer_sizes=arq, max_iter=15, random_state=SEED)
        t0 = time.perf_counter(); m.fit(X_tr, y_tr); seg = time.perf_counter() - t0
        filas.append({"arquitectura": str(arq), "parámetros": sum(w.size for w in m.coefs_) + sum(b.size for b in m.intercepts_),
                      "segundos": round(seg, 1), "accuracy val": accuracy_score(y_ev, m.predict(X_ev))})
    return pd.DataFrame(filas).set_index("arquitectura")


def permutacion_mlp(X_tr, y_tr, X_ev, y_ev):
    y_tr, y_ev = a_enteros(y_tr, y_ev); perm = permutacion_fija(X_tr.shape[1]); res = {}
    for nombre, (a, b) in {"original": (X_tr, X_ev), "píxeles permutados": (X_tr[:, perm], X_ev[:, perm])}.items():
        m = MLPClassifier(hidden_layer_sizes=(256,), early_stopping=True, max_iter=200, random_state=SEED).fit(a, y_tr)
        res[nombre] = accuracy_score(y_ev, m.predict(b))
    return res


costo = medir_costo_mlp(X_pix[m_tr], df.loc[m_tr, "actor"], X_pix[m_va], df.loc[m_va, "actor"])
display(costo)
perm_mlp = {t: permutacion_mlp(X_pix[m_tr], df.loc[m_tr, t], X_pix[m_va], df.loc[m_va, t]) for t in ("actor", "emocion")}
display(pd.DataFrame(perm_mlp).T.add_prefix("accuracy val · "))

**Acta:** si el MLP rinde prácticamente igual con los píxeles destruidos, no está viendo una imagen sino 4.096 números. Requisitos para la herramienta siguiente: **capas convolucionales** (comparten pesos sobre vecindades de píxeles y dependen de la estructura espacial; lo comprobamos en la sección 7.1), **GPU** (el costo en CPU no escala) y **callbacks** para controlar el entrenamiento (early stopping, mejores pesos).

## 6. Pipeline `tf.data`, aumentación y herramientas de entrenamiento

Patrón del curso: `map(leer) → cache → shuffle (solo train) → batch → prefetch`. Las imágenes se cachean como `uint8` (≈400 MB) y llegan al modelo en [0, 255]: escalado y aumentación van **como capas dentro del modelo** (clase 17), así el modelo lleva su propio preprocesamiento y en inferencia no se le puede alimentar distinto.

| Transformación | ¿Válida para rostros? | Argumento |
|---|---|---|
| Flip horizontal | Sí | Una cara espejada es la misma persona con el mismo gesto |
| Rotación ±8° | Sí | Las cabezas se inclinan en los videos |
| Zoom ±10 % / traslación ±5 % | Sí | Los recortes no están perfectamente centrados |
| Brillo / contraste ±15 % | Sí | Las fuentes tienen iluminación muy distinta |
| Flip vertical | **No** | Produce caras imposibles |
| Rotaciones grandes / recortes agresivos | **No** | Fabrican poses imposibles o tapan cejas y boca, que son la señal de la emoción |

**Desbalance:** pesos por clase *balanced* (`n / (K · n_c)`) en la pérdida.

In [ ]:
# 6.1 Pipeline tf.data
AUTOTUNE = tf.data.AUTOTUNE


def codificar(serie, clases):
    indice = {c: i for i, c in enumerate(clases)}
    return serie.map(indice).to_numpy(dtype=np.int32)


def _leer_imagen(ruta, y):
    img = tf.io.decode_image(tf.io.read_file(ruta), channels=3, expand_animations=False)
    img = tf.cast(tf.round(tf.image.resize(img, (IMG_SIZE, IMG_SIZE))), tf.uint8)
    img.set_shape((IMG_SIZE, IMG_SIZE, 3))
    return img, y


def _permutador(perm):
    # Permutación fija de posiciones espaciales (experimento del NB11); los 3 canales viajan juntos.
    p = tf.constant(perm, dtype=tf.int32)
    return lambda img, y: (tf.reshape(tf.gather(tf.reshape(img, (-1, 3)), p), (IMG_SIZE, IMG_SIZE, 3)), y)


def crear_dataset(rutas, y, entrenamiento, permutacion=None):
    ds = tf.data.Dataset.from_tensor_slices((np.asarray(rutas, dtype=str), np.asarray(y)))
    ds = ds.map(_leer_imagen, num_parallel_calls=AUTOTUNE)
    if permutacion is not None:
        ds = ds.map(_permutador(permutacion), num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    if entrenamiento:
        ds = ds.shuffle(len(rutas), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)


def datasets_por_split(df, tarea, permutacion=None):
    clases = sorted(df.loc[df["split"] == "train", tarea].unique())   # vocabulario fijo, tomado de train
    ys = {s: codificar(df.loc[df["split"] == s, tarea], clases) for s in SPLITS}
    ds = {s: crear_dataset(df.loc[df["split"] == s, "ruta"], ys[s], s == "train", permutacion) for s in SPLITS}
    return ds, clases, ys


def pesos_por_clase(y, k):
    conteo = np.bincount(y, minlength=k)
    return {i: float(len(y) / (k * c)) for i, c in enumerate(conteo) if c}

In [ ]:
# 6.2 Aumentación y preprocesamiento como capas
def capas_aumentacion():
    return keras.Sequential([
        layers.RandomFlip("horizontal", seed=SEED),
        layers.RandomRotation(0.022, fill_mode="nearest", seed=SEED),        # 0.022 x 360° ≈ 8°
        layers.RandomZoom(0.10, fill_mode="nearest", seed=SEED),
        layers.RandomTranslation(0.05, 0.05, fill_mode="nearest", seed=SEED),
        layers.RandomBrightness(0.15, value_range=(0, 255), seed=SEED),
        layers.RandomContrast(0.15, seed=SEED),
    ], name="aumentacion")


@keras.saving.register_keras_serializable(package="tp_g6")
class PreprocesoResNet50(layers.Layer):
    # Cada base preentrenada exige SU preprocesamiento (NB13): RGB->BGR y resta de la media de ImageNet.
    def call(self, x):
        return keras.applications.resnet50.preprocess_input(x)


# Una imagen de train y 7 versiones aumentadas
aug = capas_aumentacion()
img = np.asarray(Image.open(df.loc[df["split"] == "train", "ruta"].iloc[0]).convert("RGB"), dtype="float32")
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
axes[0].imshow(img.astype("uint8")); axes[0].set_title("original")
for k in range(1, 8):
    axes[k].imshow(np.clip(aug(img[None], training=True)[0].numpy(), 0, 255).astype("uint8")); axes[k].set_title(f"aumentada {k}")
for a in axes:
    a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 6.3 Entrenamiento y evaluación (se usan para todos los modelos)
def compilar(modelo, lr):
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])


def entrenar(modelo, ds_tr, ds_va, lr, epocas, paciencia, pesos=None, desde=0):
    # Toda decisión de cuándo parar se toma contra VALIDACIÓN, restaurando los mejores pesos.
    compilar(modelo, lr)
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=paciencia, restore_best_weights=True, start_from_epoch=desde),
          keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(2, paciencia // 2), min_lr=1e-7)]
    return modelo.fit(ds_tr, validation_data=ds_va, epochs=epocas, class_weight=pesos, callbacks=cb, verbose=2)


def contar_parametros(modelo):
    entrenables = int(sum(np.prod(w.shape) for w in modelo.trainable_weights))
    return {"total": modelo.count_params(), "entrenables": entrenables}


def graficar_curvas(historias, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8)); x0 = 0
    for nombre, h in historias.items():
        ep = np.arange(len(h.history["loss"])) + x0
        for ax, met in zip(axes, ("loss", "accuracy")):
            ax.plot(ep, h.history[met], label=f"train {nombre}")
            ax.plot(ep, h.history[f"val_{met}"], "--", label=f"val {nombre}")
        x0 += len(h.history["loss"])
    for ax, t in zip(axes, ("Pérdida", "Accuracy")):
        ax.set_title(f"{titulo} — {t}"); ax.set_xlabel("época"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()


@dataclass
class Experimento:
    nombre: str
    tarea: str
    modelo: object
    clases: list
    ds: dict
    ys: dict
    historias: dict = field(default_factory=dict)
    info: dict = field(default_factory=dict)
    val: dict = field(default_factory=dict)

    def evaluar_val(self):
        self.val = metricas(self.ys["val"], self.modelo.predict(self.ds["val"], verbose=0).argmax(1))
        return self.val

## 7. CNN propia desde cero

Anatomía del NB11 adaptada a 224×224: 4 bloques `[Conv 3×3 → Conv 3×3 → BatchNorm → MaxPool]` con filtros 32 → 64 → 128 → 256 (cada bloque reduce la resolución a la mitad y duplica los filtros: menos *dónde*, más *qué*), y cierre `GlobalAveragePooling → Dense(128) → Dropout(0,5) → softmax`.

- **GlobalAveragePooling y no Flatten:** con Flatten la densa recibiría ~50.000 entradas (millones de parámetros para ~1.900 imágenes = memorizar).
- **BatchNorm:** estabiliza el entrenamiento desde cero con lotes chicos.
- **Regularización:** aumentación + Dropout + L2 (1e-4) + early stopping (paciencia 8) sobre `val_loss`.
- **Expectativa (NB14):** con ~90 imágenes por actor, una red desde cero queda lejos de la transferencia. Es el **experimento de control** que propusimos en el TP1.

In [ ]:
# 7.1 Arquitectura y entrenamiento de la CNN propia
def cnn_propia(n_clases, filtros=(32, 64, 128, 256), dropout=0.5, densa=128, l2=1e-4, aumentar=True, nombre="cnn_propia"):
    fijar_semillas()
    reg = keras.regularizers.l2(l2)
    entrada = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = capas_aumentacion()(entrada) if aumentar else entrada
    x = layers.Rescaling(1 / 255)(x)
    for i, f in enumerate(filtros, 1):
        x = layers.Conv2D(f, 3, padding="same", activation="relu", kernel_regularizer=reg, name=f"b{i}_conv1")(x)
        x = layers.Conv2D(f, 3, padding="same", activation="relu", kernel_regularizer=reg, name=f"b{i}_conv2")(x)
        x = layers.BatchNormalization(momentum=0.9, name=f"b{i}_bn")(x)
        x = layers.MaxPooling2D(name=f"b{i}_pool")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(layers.Dense(densa, activation="relu", kernel_regularizer=reg)(x))
    return keras.Model(entrada, layers.Dense(n_clases, activation="softmax")(x), name=nombre)


def experimento_cnn(tarea, epocas=60, lr=1e-3, paciencia=10):
    ds, clases, ys = datasets_por_split(df, tarea)
    m = cnn_propia(len(clases), nombre=f"cnn_{tarea}")
    h = entrenar(m, ds["train"], ds["val"], lr, epocas, paciencia, pesos_por_clase(ys["train"], len(clases)), desde=10)
    e = Experimento("CNN propia", tarea, m, clases, ds, ys, {"entrenamiento": h},
                    {**contar_parametros(m), "épocas": len(h.history["loss"])})
    e.evaluar_val()
    return e


exps = {}
for tarea in ("actor", "emocion"):
    t0 = time.time()
    exps[("cnn", tarea)] = e = experimento_cnn(tarea)
    graficar_curvas(e.historias, f"CNN propia — {tarea}")
    print(f"{tarea}: VAL accuracy={e.val['accuracy']:.3f} · F1 macro={e.val['f1_macro']:.3f} · "
          f"{e.info['total']:,} parámetros · {e.info['épocas']} épocas · {time.time() - t0:.0f} s")

### 7.2 La revancha de la permutación (NB11)
Misma CNN, misma semilla, pero con los píxeles permutados (sin aumentación, porque rotar o espejar una imagen permutada no tiene sentido). Si la CNN **sí** usa la estructura espacial, su desempeño se tiene que desplomar: lo opuesto a lo que le pasó al MLP.

In [ ]:
# 7.2 CNN con píxeles permutados (identidad)
ds_p, clases_p, ys_p = datasets_por_split(df, "actor", permutacion=permutacion_fija(IMG_SIZE * IMG_SIZE))
cnn_perm = cnn_propia(len(clases_p), aumentar=False, nombre="cnn_permutada")
entrenar(cnn_perm, ds_p["train"], ds_p["val"], 1e-3, 60, 10, pesos_por_clase(ys_p["train"], len(clases_p)), desde=10)
acc_cnn_perm = metricas(ys_p["val"], cnn_perm.predict(ds_p["val"], verbose=0).argmax(1))["accuracy"]
display(pd.DataFrame({
    "accuracy val · original": [perm_mlp["actor"]["original"], exps[("cnn", "actor")].val["accuracy"]],
    "accuracy val · píxeles permutados": [perm_mlp["actor"]["píxeles permutados"], acc_cnn_perm]},
    index=["MLP scikit-learn", "CNN propia"]))

## 8. Transferencia: ResNet50 preentrenada en ImageNet

**Por qué ResNet50** (propuesta del TP1): nuestros recortes ya son 224×224×3, su **entrada nativa**; con ~90 imágenes por clase no alcanza para aprender filtros desde cero; las primeras capas de ResNet50 detectan bordes y texturas genéricas que sirven tal cual, y las últimas (conv5) codifican semántica de ImageNet: son las que hay que adaptar a rostros.

**Secuencia del NB15** (el orden *es* el contenido):
- **Fase 1:** base congelada; solo se entrena la cabeza nueva (`GAP → Dropout(0,3) → softmax`), Adam 1e-3. Una cabeza aleatoria produce gradientes grandes que, con la base libre, destruirían lo aprendido.
- **Fase 2:** se descongela solo el bloque **conv5** (las BatchNorm siguen congeladas y en modo inferencia con `training=False`), se **recompila** con Adam **1e-5** (100 veces menor).

In [ ]:
# 8.1 ResNet50 en dos fases
def resnet50_transferencia(n_clases, nombre):
    fijar_semillas()
    base = keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    entrada = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = PreprocesoResNet50()(capas_aumentacion()(entrada))
    x = base(x, training=False)          # BatchNorm en modo inferencia aunque luego se descongele
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(entrada, layers.Dense(n_clases, activation="softmax")(x), name=nombre), base


def descongelar_conv5(base, primera="conv5_block1_1_conv"):
    base.trainable = True
    corte = [l.name for l in base.layers].index(primera)
    for i, capa in enumerate(base.layers):
        capa.trainable = i >= corte and not isinstance(capa, layers.BatchNormalization)


def experimento_resnet(tarea, ep1=20, lr1=1e-3, ep2=20, lr2=1e-5, paciencia=5):
    ds, clases, ys = datasets_por_split(df, tarea)
    pesos = pesos_por_clase(ys["train"], len(clases))
    m, base = resnet50_transferencia(len(clases), f"resnet50_{tarea}")
    p1 = contar_parametros(m)
    h1 = entrenar(m, ds["train"], ds["val"], lr1, ep1, paciencia, pesos)
    e = Experimento("ResNet50 fase 1 (base congelada)", tarea, m, clases, ds, ys, {"fase 1": h1},
                    {"total": p1["total"], "entrenables fase 1": p1["entrenables"], "épocas fase 1": len(h1.history["loss"])})
    e.info["val fase 1"] = dict(e.evaluar_val())
    descongelar_conv5(base)                                    # + recompilar dentro de entrenar()
    h2 = entrenar(m, ds["train"], ds["val"], lr2, ep2, paciencia, pesos)
    e.nombre = "ResNet50 fase 2 (fine-tuning conv5)"
    e.historias["fase 2"] = h2
    e.info.update({"entrenables fase 2": contar_parametros(m)["entrenables"], "épocas fase 2": len(h2.history["loss"])})
    e.evaluar_val()
    return e


for tarea in ("actor", "emocion"):
    t0 = time.time()
    exps[("resnet", tarea)] = e = experimento_resnet(tarea)
    graficar_curvas(e.historias, f"ResNet50 — {tarea}")
    v1 = e.info["val fase 1"]
    print(f"{tarea}: VAL fase 1 acc={v1['accuracy']:.3f} F1m={v1['f1_macro']:.3f}  →  fase 2 acc={e.val['accuracy']:.3f} F1m={e.val['f1_macro']:.3f}")
    print(f"   entrenables: fase 1 {e.info['entrenables fase 1']:,} · fase 2 {e.info['entrenables fase 2']:,} de {e.info['total']:,} · {time.time() - t0:.0f} s")

### 8.2 Comparación en validación (acá se toman las decisiones)

In [ ]:
filas = []
for tarea in ("actor", "emocion"):
    for nombre, fila in resultados_baseline[tarea].iterrows():
        filas.append({"tarea": tarea, "modelo": nombre, "accuracy": fila["accuracy"], "f1_macro": fila["f1_macro"]})
    filas.append({"tarea": tarea, "modelo": "CNN propia", **exps[("cnn", tarea)].val})
    filas.append({"tarea": tarea, "modelo": "ResNet50 fase 1 (base congelada)", **exps[("resnet", tarea)].info["val fase 1"]})
    filas.append({"tarea": tarea, "modelo": "ResNet50 fase 2 (fine-tuning conv5)", **exps[("resnet", tarea)].val})
comparacion_val = pd.DataFrame(filas).set_index(["tarea", "modelo"])
display(comparacion_val.style.format("{:.3f}").highlight_max(axis=0, props="font-weight:bold"))

## 9. Evaluación final en test — una sola vez

Todas las decisiones anteriores se tomaron con validación. Ahora, y solo ahora, se evalúan en test los modelos finales de cada tarea. Esta celda no se vuelve a correr para "probar otra cosa".

In [ ]:
# 9.1 Test
resultados_test = {}
for clave, e in exps.items():
    prob = e.modelo.predict(e.ds["test"], verbose=0)
    resultados_test[clave] = {"prob": prob, "metricas": metricas(e.ys["test"], prob.argmax(1)),
                              "rutas": df.loc[df["split"] == "test", "ruta"].to_numpy()}
tabla_test = pd.DataFrame([{"tarea": t, "modelo": exps[(m, t)].nombre, **resultados_test[(m, t)]["metricas"]}
                           for (m, t) in exps]).set_index(["tarea", "modelo"])
display(tabla_test.style.format("{:.3f}"))
print(f"Huella del dataset: {HUELLA} · semilla: {SEED}")

In [ ]:
# 9.2 Reporte por clase y matriz de confusión del mejor modelo de cada tarea
def pares_confundidos(y, prob, clases, n=5):
    cm = confusion_matrix(y, prob.argmax(1), labels=range(len(clases))); np.fill_diagonal(cm, 0)
    idx = np.dstack(np.unravel_index(np.argsort(-cm.ravel()), cm.shape))[0][:n]
    return pd.DataFrame([{"real": clases[i], "predicho": clases[j], "casos": int(cm[i, j])} for i, j in idx if cm[i, j]])


for tarea in ("actor", "emocion"):
    clave = max([("cnn", tarea), ("resnet", tarea)], key=lambda k: exps[k].val["f1_macro"])   # elegido por VALIDACIÓN
    e, r = exps[clave], resultados_test[clave]
    print(f"\n===== {tarea.upper()} · {e.nombre} (elegido por F1 macro en validación) =====")
    print(classification_report(e.ys["test"], r["prob"].argmax(1), labels=range(len(e.clases)), target_names=e.clases, zero_division=0))
    lado = max(5, 0.45 * len(e.clases) + 2)
    fig, ax = plt.subplots(figsize=(lado, lado))
    cm = confusion_matrix(e.ys["test"], r["prob"].argmax(1), labels=range(len(e.clases)), normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=e.clases).plot(ax=ax, cmap="Blues", colorbar=False, values_format=".2f", xticks_rotation=90)
    ax.set_title(f"{e.nombre} — {tarea} (test, normalizada por fila)"); plt.tight_layout(); plt.show()
    display(pares_confundidos(e.ys["test"], r["prob"], e.clases))

In [ ]:
# 9.3 Errores con mayor confianza: ¿razonables o absurdos?
def peores_errores(rutas, y, prob, clases, titulo, n=12):
    pred = prob.argmax(1); err = np.where(pred != y)[0]
    peores = err[np.argsort(-prob[err, pred[err]])][:n]
    fig, axes = plt.subplots(2, 6, figsize=(14, 5.6))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), peores):
        ax.imshow(Image.open(rutas[i]))
        ax.set_title(f"real: {clases[y[i]]}\npred: {clases[pred[i]]} ({prob[i, pred[i]]:.0%})", fontsize=8, color="darkred")
    fig.suptitle(titulo); plt.tight_layout(); plt.show()


for tarea in ("actor", "emocion"):
    e, r = exps[("resnet", tarea)], resultados_test[("resnet", tarea)]
    peores_errores(r["rutas"], e.ys["test"], r["prob"], e.clases, f"Peores errores — ResNet50 — {tarea}")

## 10. Interpretación: ¿el modelo mira la cara?

Mapa de oclusión (NB12): se tapa cada región con un parche gris y se mide cuánto cae la probabilidad de la clase correcta; en rojo, las regiones decisivas. Es la prueba del riesgo que declaramos en el TP1: como cada actor sale de pocas fuentes, un modelo de identidad podría estar reconociendo **el fondo o la serie**. Para emoción, esperamos la evidencia en **cejas, ojos y boca**.

In [ ]:
# 10.1 Mapas de oclusión
def mapa_oclusion(modelo, imagen, clase, parche=32, paso=16, gris=127.5):
    pos = list(range(0, IMG_SIZE - parche + 1, paso)); lote = []
    for y in pos:
        for x in pos:
            t = imagen.copy(); t[y:y + parche, x:x + parche] = gris; lote.append(t)
    p0 = float(modelo.predict(imagen[None], verbose=0)[0, clase])
    caidas = (p0 - modelo.predict(np.stack(lote), batch_size=64, verbose=0)[:, clase]).reshape(len(pos), len(pos))
    mapa, cuenta = np.zeros((IMG_SIZE, IMG_SIZE)), np.zeros((IMG_SIZE, IMG_SIZE))
    for a, y in enumerate(pos):
        for b, x in enumerate(pos):
            mapa[y:y + parche, x:x + parche] += caidas[a, b]; cuenta[y:y + parche, x:x + parche] += 1
    return mapa / np.maximum(cuenta, 1), p0


for tarea in ("actor", "emocion"):
    e, r = exps[("resnet", tarea)], resultados_test[("resnet", tarea)]
    bien = np.where(r["prob"].argmax(1) == e.ys["test"])[0]
    elegidas = np.random.default_rng(SEED).choice(bien, size=min(5, len(bien)), replace=False)
    fig, axes = plt.subplots(2, len(elegidas), figsize=(3 * len(elegidas), 6.2))
    for k, i in enumerate(elegidas):
        img = np.asarray(Image.open(r["rutas"][i]).convert("RGB"), dtype="float32")
        mapa, p = mapa_oclusion(e.modelo, img, int(e.ys["test"][i]))
        axes[0, k].imshow(img.astype("uint8")); axes[0, k].set_title(f"{e.clases[e.ys['test'][i]]} (p={p:.2f})", fontsize=9)
        axes[1, k].imshow(img.astype("uint8")); axes[1, k].imshow(mapa, cmap="jet", alpha=0.5)
        axes[0, k].axis("off"); axes[1, k].axis("off")
    fig.suptitle(f"Oclusión — ResNet50 — {tarea} (rojo = región decisiva)"); plt.tight_layout(); plt.show()

## 11. Conclusiones y limitaciones

_Se completa con los números de esta corrida (secciones 8.2 y 9)._

**Declaración de uso de IA generativa:** se usó un asistente de IA (Claude) para auditar el dataset, proponer la estructura del código y redactar borradores de la documentación. Todas las decisiones fueron revisadas por el grupo y cualquier integrante puede explicar cada celda.